In [1]:
import pandas as pd
import numpy as np
import utils
import os
from pathlib import Path

In [ ]:
def getPlayersMatch(df_events):
  players = {}
  for id, player in zip(df_events.playerId, df_events.playerName):
    if not np.isnan(id):
      players[id] = player

  return players

In [2]:
def translateRelatedEvent(satisfiedEvents):
  file = 'Dataset/WhoScored/event_metadata.json'
  event_metadata = utils.readJson(file)
  event_metadata = {value : key for key,value in event_metadata.items()}
  return [event_metadata[x] for x in satisfiedEvents]

def misureDistance(x1, y1, x2, y2):
  return np.sqrt((x1-x2)**2 + (y1-y2)**2)

def traduciEvento(events_dict,event, lang='en'):
  #events_dict = readJson( f'{GDRIVE_THESIS_DIR}/action_translations.json')
  events = events_dict[lang]['events']
  related_events = events_dict[lang]['relatedEvents']
  current_event = event['displayName']
  translated_event = events[current_event]
  #current_related_events = translateRelatedEvent(satisfiedEvents)
  #translated_related_events = [related_events[x] for x in current_related_events if related_events[x] != '']
  #str_related_events = f" ({', '.join(translated_related_events).strip(', ')})" if len(translated_related_events) > 0 else ""
  return translated_event

def traduciEventiSupplementari(events_dict, satisfiedEvents, qualifiers, lang='en'):
  qualifiers_dict = events_dict[lang]['qualifiers']
  related_events = events_dict[lang]['relatedEvents']
  current_related_events = translateRelatedEvent(satisfiedEvents)
  qualifierNames = [x['type']['displayName'] for x in qualifiers]
  translated_qualifiers = [qualifiers_dict[x] for x in qualifierNames if qualifiers_dict[x] != '']
  translated_related_events = [related_events[x] for x in current_related_events if related_events[x] != ''] + translated_qualifiers
  str_related_events = f" ({', '.join(translated_related_events).strip(', ')})" if len(translated_related_events) > 0 else ""

  return str_related_events


def componiFrase(row, events_dict, lang='en'):
    #esito = 'con successo' if row['outcomeType']['value'] == 1 else 'fallendo'
    start_position = detectFieldBin(row['x'], row['y'])
    prep_from = 'da' if lang == 'it' else 'from'
    prep_to = 'a' if lang == 'it' else 'to'
    if start_position != '':

      start_position = f"{prep_from} {start_position}"
    else:
      start_position = ''

    if np.isnan(row['endX']) or np.isnan(row['endY']):
      end_position = ''
    else:
      end_position = f"{prep_to} {detectFieldBin(row['endX'], row['endY'])}"
    '''
    if pd.isna(row['playerName']):
      player=''
    else:
      player = row['playerName']

    if pd.isna(row['teamName']):
      team =''
    else:
      team = f"({row['teamName']})"
  '''
    #frase = f"{player} {team} {traduciEvento(events_dict, row['type'])} {start_position}{end_position} {traduciEventiSupplementari(events_dict, row['satisfiedEventsTypes'], row['qualifiers'])}"
    frase = f"{traduciEvento(events_dict, row['type'], lang)} {start_position} {end_position} {traduciEventiSupplementari(events_dict, row['satisfiedEventsTypes'], row['qualifiers'], lang)}"
    return frase.strip()

def detectFieldBin(x , y, n_binx: int = 5, n_biny: int = 5, lang: str = 'en'):
    if x == 0 and y == 0:
      return ''
    if lang == 'it':
      x_text = {1: 'difesa', 2: 'trequarti difensiva', 3: 'centrocampo', 4: 'trequarti offensiva', 5: 'attacco'}
      y_text = {1: 'fascia destra', 2: 'centro destra', 3: 'centrale', 4: 'centro sinistra', 5: 'fascia sinistra'}
    else:
      x_text = {1: 'defensive', 2: 'defensive third', 3: 'midfield', 4: 'attacking third', 5: 'attack'}
      y_text = {1: 'right flank', 2: 'right center', 3: 'central', 4: 'left center', 5: 'left flank'}

    field_length_x = 100
    field_length_y = 100
    bin_x_width, bin_y_width = np.ceil(field_length_x / n_binx), np.ceil(field_length_y / n_biny)
    bin_x = int((x - 1) / bin_x_width) + 1
    bin_y = int((y - 1) / bin_y_width) + 1
    return f"{x_text[bin_x]}, {y_text[bin_y]}"


In [4]:
def componiTextPlayerMatch(player_match: dict) -> str:
    text = ''
    i = 0
    for id, action in player_match['text'].items():
        if i > 0:
            if int(id) == i+1:
                text = text +', ' + action
            else:
                text = text +'. ' + action
        else:
            text = action
        i = int(id)

    return text

In [2]:
import os
src_dir = 'Dataset/WhoScored'
src_dir.split('/')[-1]

'WhoScored'

In [3]:
src_dir = os.path.join('Dataset', 'WhoScored')
lang='en'
events_dict = utils.readJson( f'action_translations.json')
new_related_events = set()
new_qualifiers = set()
new_events = set()


src_dir_path = Path(src_dir)
src_dir_league = [d.name for d in src_dir_path.iterdir() if d.is_dir() and '-' in d.name]
for league_dir in src_dir_league:
    league_dir_abs = os.path.join(src_dir, league_dir)
    tgt_league_dir_abs = league_dir_abs.replace('WhoScored','Events2TextEN')
    os.makedirs(tgt_league_dir_abs, exist_ok=True)
    league_dir_path = Path(league_dir_abs)
    src_dir_league_season = [d.name for d in league_dir_path.iterdir() if d.is_dir() and '-' in d.name]
    for season in src_dir_league_season:
        src_dir_full = os.path.join(league_dir_abs, season)
        tgt_dir_full = os.path.join(tgt_league_dir_abs,season)
        os.makedirs(tgt_dir_full, exist_ok=True)
        input_files = os.listdir(src_dir_full)
        for match in input_files:
            match_events = []
            file = os.path.join(src_dir_full, match)
            df_events = pd.read_json(file)

            for idx, row in df_events.iterrows():
                '''
                qualifiers = [x['type']['displayName'] for x in row['qualifiers']]
                event = row['type']['displayName']
                if event not in events_dict['events'].keys():
                    new_events.add(event)
                for q in qualifiers:
                    if q not in events_dict['qualifiers'].keys():
                        new_qualifiers.add(q)
                related_events = translateRelatedEvent(row['satisfiedEventsTypes'])'
                for rev in related_events:
                    if rev not in events_dict['relatedEvents'].keys():
                        new_related_events.add(rev)
                '''
                match_events.append({'playerId': row['playerId'], 'playerName': row['playerName'],'teamId': row['teamId'], 'team': row['teamName'], 'text': componiFrase(row, events_dict)})
                
            utils.writeJson(match_events, os.path.join(tgt_dir_full, match))


In [7]:
def getPlayersMatch(df_events):
    teams = {}
    try:
        for id, name in zip(df_events.teamId, df_events.teamName):
            if not pd.isna(name):
                teams[id] = name
    except:
        for id, name in zip(df_events.teamId, df_events.team):
            if not pd.isna(name):
                teams[id] = name

    teams_list = {x: { 'name': teams[x], 'players': {}} for x in teams.keys()}

    for id, row in df_events.iterrows():
        teamId = row['teamId']
        id = row['playerId']
        if not np.isnan(id):
            teams_list[teamId]['players'][id] = row['playerName']

    return teams_list

def aggiungiPlayersFromMatch(teams_list_all, teams_list_match):
    for team in teams_list_match.keys():
        if team not in teams_list_all:
            teams_list_all[team] = teams_list_match[team]
        else:
            teams_list_all[team]['players'] = teams_list_all[team]['players'] | teams_list_match[team]['players'] 


Numero di eventi totali: 20.846.939 <br>
Numero di partite totali: 13.331

In [ ]:
src_dir = os.path.join('Dataset', 'WhoScored')
tgt_dir = os.path.join('Dataset', 'Events2Text')

def getPlayersTeamListPerSeason(src_dir, tgt_dir):
    src_dir_path = Path(src_dir)
    src_dir_league = [d.name for d in src_dir_path.iterdir() if d.is_dir() and '-' in d.name]
    for league_dir in src_dir_league:
        league_dir_abs = os.path.join(src_dir, league_dir)
        league_dir_path = Path(league_dir_abs)
        src_dir_league_season = [d.name for d in league_dir_path.iterdir() if d.is_dir() and '-' in d.name]
        for season in src_dir_league_season:
            src_dir_full = os.path.join(league_dir_abs, season)
            input_files = os.listdir(src_dir_full)
            teams_list = {}
            for match in input_files:
                file = os.path.join(src_dir_full, match)
                df_events = pd.read_json(file)
                match_players = getPlayersMatch(df_events)
                aggiungiPlayersFromMatch(teams_list, match_players)
            
            utils.writeJson(teams_list, os.path.join(tgt_dir, f'{league_dir}_{season}_teams.json'))
            

In [ ]:
src_dir = os.path.join('Dataset', 'Events2Text')

def splitDocsPerPlayer(src_dir):
    out_dir = os.path.join(src_dir, 'PlayerDocs')
    src_dir_path = Path(src_dir)
    src_dir_league = [d.name for d in src_dir_path.iterdir() if d.is_dir() and '-' in d.name]
    for league_dir in src_dir_league:
        league_dir_abs = os.path.join(src_dir, league_dir)
        league_dir_path = Path(league_dir_abs)
        src_dir_league_season = [d.name for d in league_dir_path.iterdir() if d.is_dir() and '-' in d.name]
        for season in src_dir_league_season:
            src_dir_full = os.path.join(league_dir_abs, season)
            input_files = os.listdir(src_dir_full)
            for match in input_files:
                file = os.path.join(src_dir_full, match)
                df_events = pd.read_json(file)
                team_players = getPlayersMatch(df_events)
                for team in team_players.keys():
                    ply = team_players[team]['players']
                    for p in ply.keys():
                        df_p_events = df_events[df_events.playerId == p]
                        out_dir_p = os.path.join(out_dir, str(p))
                        os.makedirs(out_dir_p, exist_ok=True)
                        out_dir_p_season = os.path.join(out_dir_p, season)
                        os.makedirs(out_dir_p_season, exist_ok = True)
                        df_p_events.to_json(os.path.join(out_dir_p_season, match))
        


In [ ]:
src_dir = os.path.join('Dataset', 'WhoScored')

teams_list = {}
for match in os.listdir(src_dir):
    df_events = pd.read_json(os.path.join(src_dir, match))
    match_players = getPlayersMatch(df_events)
    aggiungiPlayersFromMatch(teams_list, match_players)

teams_list

In [ ]:
#print("calcio d'inizio")
events_dict = utils.readJson( f'{src_dir}/action_translations.json')

new_related_events = set()
new_qualifiers = set()
new_events = set()
for match in os.listdir(src_dir_league_season):
    match_events = []
    file = os.path.join(src_dir_league_season, match)
    df_events = pd.read_json(file)

    for idx, row in df_events.iterrows():
        event = row['type']['displayName']
        events = events_dict['events']
        if event not in events.keys():
            new_events.add(event)
        match_events.append({'playerId': row['playerId'], 'playerName': row['playerName'], 'team': row['teamName'], 'text': componiFrase(row, events_dict)})

    #utils.writeJson(match_events, os.path.join(tgt_dir_league_season, match))
#print('fischio finale')
new_events

set()

<h1>Salvare tutto il dataset in un unico json</h1>

In [ ]:
src_dir = 'Dataset\\Events2Text\\PlayerDocs'
tgt_dir = 'Dataset\\Events2Text'
entire_dataset = []
j=0
for p in os.listdir(src_dir):
    src_dir_p = os.path.join(src_dir, p)
    for s in os.listdir(src_dir_p):
        src_dir_p_s = os.path.join(src_dir_p, s)
        for match in os.listdir(src_dir_p_s):
            player_match = utils.readJson(os.path.join(src_dir_p_s, match))
            teamId = list(player_match['teamId'].values())[0]
            team = list(player_match['team'].values())[0]
            playerId = list(player_match['playerId'].values())[0]
            playerName = list(player_match['playerName'].values())[0]
            record = dict(season=s, playerId=playerId, playerName=playerName, teamId=teamId, teamName=team)
            record['text'] = componiTextPlayerMatch(player_match)
            record['match'] = match
            entire_dataset.append(record)

utils.writeJson(entire_dataset, os.path.join(tgt_dir, 'player2vec_dataset.json'))

In [4]:
def componiTextPlayerMatch(player_match: dict) -> str:
    text = ''
    i = 0
    for id, action in player_match['text'].items():
        if i > 0:
            if int(id) == i+1:
                text = text +', ' + action
            else:
                text = text +'. ' + action
        else:
            text = action
        i = int(id)

    return text

In [15]:
src_dir = 'Dataset/Events2TextEN'
entire_dataset = []

for x in os.listdir(src_dir):
    league_dir = f'{src_dir}/{x}' 
    if os.path.isdir(league_dir) and '-' in x:
        for season in os.listdir(league_dir):
            league_season_dir = f'{league_dir}/{season}' 
            print(league_season_dir)
            for match_json in os.listdir(league_season_dir):
                match = match_json[:-5]
                df_match = pd.read_json(f'{league_season_dir}/{match_json}')
                players_team = getPlayersMatch(df_match)
                for team in players_team.keys():
                    players = players_team[team]['players']
                    for p in players.keys():
                        df_match_player = df_match[df_match.playerId == p]
                        text = componiTextPlayerMatch(df_match_player)
                        teamName = players_team[team]['name']
                        playerName = players[p]
                        record = dict(season=season, playerId=p, playerName=playerName, teamId=team, teamName=teamName, text=text, match=match)
                        entire_dataset.append(record)

print(f'Numero di record caricati: {len(entire_dataset)}')
utils.writeJson(entire_dataset, f'{src_dir}/player2vec_dataset_en.json')

Dataset/Events2TextEN/Europa-Champions-League/2019-2020
Dataset/Events2TextEN/Europa-Champions-League/2020-2021
Dataset/Events2TextEN/Europa-Champions-League/2021-2022
Dataset/Events2TextEN/Europa-Champions-League/2022-2023
Dataset/Events2TextEN/Europa-Champions-League/2023-2024
Dataset/Events2TextEN/Europa-Europa-League/2019-2020
Dataset/Events2TextEN/Europa-Europa-League/2020-2021
Dataset/Events2TextEN/Europa-Europa-League/2021-2022
Dataset/Events2TextEN/Europa-Europa-League/2022-2023
Dataset/Events2TextEN/Europa-Europa-League/2023-2024
Dataset/Events2TextEN/Francia-Ligue-1/2019-2020
Dataset/Events2TextEN/Francia-Ligue-1/2020-2021
Dataset/Events2TextEN/Francia-Ligue-1/2021-2022
Dataset/Events2TextEN/Francia-Ligue-1/2022-2023
Dataset/Events2TextEN/Francia-Ligue-1/2023-2024
Dataset/Events2TextEN/Germania-Bundesliga/2019-2020
Dataset/Events2TextEN/Germania-Bundesliga/2020-2021
Dataset/Events2TextEN/Germania-Bundesliga/2021-2022
Dataset/Events2TextEN/Germania-Bundesliga/2022-2023
Dataset

In [45]:
set(x for x in df_match_teams.team if x != None).pop()

'Bayern'

In [4]:
src_dir = 'Dataset/Events2Text'
dataset = utils.readJson(f'{src_dir}/team2vec_datasetv2.json')
dataset[3000]

{'teamId': 304,
 'teamName': 'PSG',
 'text': "Calcio d'inizio. Passaggio bloccato  da trequarti difensiva, fascia sinistra  (passaggio bloccato da un giocatore di movimento). Tackle  da difesa, fascia sinistra  (tackle vinto), guadagna calcio d'angolo  da difesa, fascia sinistra  (guadagna calcio d'angolo, tiro fuori a sinistra). Spazzata  da difesa, centrale a trequarti difensiva, centro sinistra  (spazzata efficace, spazzata effettuata di testa, colpo di testa). Passaggio  da difesa, fascia sinistra a trequarti difensiva, fascia sinistra  (preciso, corto preciso, in avanti, verso sinistra), Recupero possesso  da trequarti difensiva, fascia sinistra, Passaggio  da trequarti difensiva, fascia sinistra a trequarti difensiva, fascia sinistra  (preciso, corto preciso, in avanti, verso destra), Dribbling  da trequarti difensiva, fascia sinistra  (dribbling perso). Contesa pallone in aria  da centrocampo, fascia destra  (duello aereo perso, duello offensivo, azione offensiva). Recupero poss

In [42]:
def getPlayersList(path: str) -> set:
    players_list = []
    #for file in os.listdir(path):
    for file in os.listdir(path):
        dict_file = utils.readJson(f'{path}/{file}')
        for row in dict_file.values():
            players = list(row['players'].keys())
            players_list = players_list + players
    players_list = [float(player) if '.' in player else int(player) for player in players_list]
    return set(players_list) - {0.0}

def getPlayersDict(path: str) -> set:
    players_list = {}
    #for file in os.listdir(path):
    for file in os.listdir(path):
        nazione = file.split('-')[0]
        dict_file = utils.readJson(f'{path}/{file}')
        for row in dict_file.values():
            if nazione != 'Europa':
                players_list = players_list | row['players']
            #players_list = players_list + players
    #players_list = [float(player) if '.' in player else int(player) for player in players_list]
    return players_list

In [48]:
players_path = 'Dataset/Events2Text/TeamPlayerList'
players_dict= getPlayersDict(players_path)
getTeamDict(players_path)

{'950': 'Reims',
 '249': 'Marseille',
 '613': 'Nice',
 '151': 'Amiens',
 '302': 'Nantes',
 '607': 'Lille',
 '315': 'Bordeaux',
 '614': 'Angers',
 '2332': 'Brest',
 '246': 'Toulouse',
 '1364': 'Dijon',
 '145': 'Saint-Etienne',
 '248': 'Monaco',
 '228': 'Lyon',
 '311': 'Montpellier',
 '313': 'Rennes',
 '245': 'Nimes',
 '304': 'PSG',
 '314': 'Metz',
 '148': 'Strasbourg',
 '146': 'Lorient',
 '309': 'Lens',
 '229': 'Troyes',
 '941': 'Clermont Foot',
 '610': 'AC Ajaccio',
 '308': 'Auxerre',
 '217': 'Le Havre',
 '37': 'Bayern',
 '47': 'Hertha Berlin',
 '1730': 'Augsburg',
 '44': 'Borussia Dortmund',
 '134': 'Borussia M.Gladbach',
 '39': 'Schalke',
 '45': 'Eintracht Frankfurt',
 '1211': 'Hoffenheim',
 '810': 'Paderborn',
 '36': 'Leverkusen',
 '7614': 'RBL',
 '796': 'Union Berlin',
 '1150': 'Fortuna Duesseldorf',
 '42': 'Werder Bremen',
 '282': 'FC Koln',
 '33': 'Wolfsburg',
 '219': 'Mainz',
 '50': 'Freiburg',
 '41': 'Stuttgart',
 '40': 'Arminia Bielefeld',
 '89': 'Greuther Fuerth',
 '109': 'Bo

In [47]:
def getTeamDict(path: str) -> set:
    team_list = {}
    #for file in os.listdir(path):
    for file in os.listdir(path):
        nazione = file.split('-')[0]
        dict_file = utils.readJson(f'{path}/{file}')
        for k, v in dict_file.items():
            if nazione != 'Europa':
                new_row = {k: v['name']}
                team_list = team_list | new_row
            #players_list = players_list + players
    #players_list = [float(player) if '.' in player else int(player) for player in players_list]
    return team_list

In [ ]:
dict_vocab = utils.readJson('action_translations.json')
vocab = []
for d in dict_vocab.values():
    vocab = vocab + list(d.values())
vocab = set(vocab) - {''}
x_text = {1: 'difesa', 2: 'trequarti difensiva', 3: 'centrocampo', 4: 'trequarti offensiva', 5: 'attacco'}
y_text = {1: 'fascia destra', 2: 'centro destra', 3: 'centrale', 4: 'centro sinistra', 5: 'fascia sinistra'}
field_vocab = [f'{x}, {y}' for x in x_text.values() for y in y_text.values()]
total_vocab = [f'{x} {y}' for x in vocab for y in field_vocab]

(207, 25, 5175)

In [ ]:
from chromadb_ingestion import PlayerEmbeddingFunction
model = PlayerEmbeddingFunction('Model_v2/Model')
vocab_embeddings = {}
for v in total_vocab:
    vocab_embeddings[v] = model(v)